In [31]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    graph_builder = StateGraph(State)

    llm = ChatOpenAI(model_name=model_name)

    def chatbot(state: State):
        return {"messages": [llm.invoke(state["messages"])]}
    
    # グラフにチャットボットノードを追加
    graph_builder.add_node("chatbot", chatbot)

    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")
    # 終了ノードの指定
    graph_builder.set_finish_point("chatbot")

    # 実行可能なステートグラフの作成
    graph = graph_builder.compile()
    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    # ソースコードを記述
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)


# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
# ソースコードを記述
graph = build_graph(MODEL_NAME)

# メインループ
# ソースコードを記述
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)


てst
こんにちは！何かお手伝いできることがありますか？


こんにちは
こんにちは！何かお手伝いできることがあれば教えてください。


ありがとうございました!
